In [1]:
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm
import os
import pandas_market_calendars as mcal
from itertools import combinations
import statsmodels.api as sm
import json
from xgboost import XGBRegressor

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.operators.repair.rounding import RoundingRepair
from multiprocessing.pool import ThreadPool
from pymoo.parallelization.starmap import StarmapParallelization
from statsmodels.tsa.stattools import coint

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb

from utils import get_data_from_cache, get_universe_by_period

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

In [2]:
universe = pd.read_csv("filtered_universe.csv")
universe = universe.dropna(subset="GICS Sector")
universe = universe[universe["GICS Sector"] != "Unknown"]
universe["start"] = pd.to_datetime(universe["start"])
universe["end"] = pd.to_datetime(universe["end"])
universe.head()

,ticker,start,end,data_length,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,AAL,2015-03-23,2024-07-08,2338,NaN,Industrials,NaN,NaN,NaN,NaN,NaN
1,AAPL,1996-01-02,2026-01-14,7558,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals","Cupertino, California",1982-11-30,320193.0,1977
2,ABT,1996-01-02,2026-01-14,7558,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800.0,1888
3,ADM,1996-01-02,2026-01-14,7558,Archer Daniels Midland,Consumer Staples,Agricultural Products & Services,"Chicago, Illinois",1957-03-04,7084.0,1902
4,ADP,1996-01-02,2026-01-14,7558,Automatic Data Processing,Industrials,Human Resource & Employment Services,"Roseland, New Jersey",1981-03-31,8670.0,1949


In [3]:
def compute_spread(series1, series2):
    model = sm.OLS(series1, sm.add_constant(series2)).fit()
    spread = model.resid
    return spread

def compute_cointegration(series1, series2):
    spread = compute_spread(series1, series2)
    adf_result = sm.tsa.adfuller(spread)
    return adf_result[1]  # p-value

def compute_half_life(spread):
    # Fit Ornstein-Uhlenbeck (AR(1) process on the spread)
    # delta_S(t) = a + lambda * S(t-1) + epsilon
    spread_lag = spread.shift().dropna()
    spread_diff = spread.diff().dropna()
    
    # Align indexes
    idx = spread_lag.index.intersection(spread_diff.index)
    if len(idx) < 10:
        return np.inf

    X = sm.add_constant(spread_lag.loc[idx])
    y = spread_diff.loc[idx]
    
    model = sm.OLS(y, X).fit()
    
    if len(model.params) > 1:
        lambda_ = model.params.iloc[1]
    else:
        lambda_ = model.params.iloc[0]
    
    # If lambda is positive or very close to 0, it means it's not mean-reverting
    if lambda_ >= -1e-8:
        return np.inf
    
    half_life = -np.log(2) / lambda_
    return half_life

In [4]:
from multiprocessing.pool import ThreadPool
from pymoo.core.problem import ElementwiseProblem
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import numpy as np

# On initialise le ThreadPool
pool = ThreadPool(processes=os.cpu_count())

class PairsTradingProblem(ElementwiseProblem):
    def __init__(self, df):
        self.df_prices = df
        self.n_stocks = df.shape[1]
        super().__init__(n_var=2, 
                         n_obj=3, 
                         n_ieq_constr=1, 
                         xl=np.array([0, 0]), 
                         xu=np.array([self.n_stocks - 1, self.n_stocks - 1]), 
                         vtype=int,
                         runner=pool.starmap, # On passe correctement le threadpool à pymoo
                         )

    def _evaluate(self, x, out, *args, **kwargs):
        idx1, idx2 = int(x[0]), int(x[1])
        out["G"] = [idx1 - idx2 + 1]  # constraint to ensure idx1 < idx2
        
        if idx1 >= idx2:
            out["F"] = [1, 0, 500]
            return
            
        # Convertir en NumPy pour accélérer drastiquement les calculs !
        series1 = self.df_prices.iloc[:, idx1].values
        series2 = self.df_prices.iloc[:, idx2].values
        
        # Calculer la corrélation ultra-rapidement avec NumPy
        corr = np.corrcoef(series1, series2)[0, 1]
        # On baisse le seuil à 0.2. Une corrélation de 0.7 sur des prix est parfois trop stricte 
        # (des actifs peuvent être cointégrés avec une corrélation prix de 0.5)
        # if corr < 0.2:
        #     out["F"] = [1, 0, 500]
        #     return

        try:
            _, pvalue, _ = coint(series1, series2)
        except:
            pvalue = 1  

        try:
            model = sm.OLS(series1, sm.add_constant(series2)).fit()
            hr = model.params[1]
            if hr <= 0:
                out["F"] = [1, 0, 500] 
                return
            spread = model.resid
        except:
            out["F"] = [1, 0, 500]
            return
            
        volatility = np.std(spread)

        try:
            spread_lag = spread[:-1]
            spread_diff = spread[1:] - spread[:-1]
            X_hl = sm.add_constant(spread_lag)
            model_hl = sm.OLS(spread_diff, X_hl).fit()
            lambda_ = model_hl.params[1] if len(model_hl.params) > 1 else model_hl.params[0]
            
            if lambda_ >= -1e-8:
                hl = 500
            else:
                hl = -np.log(2) / lambda_
            
            if np.isnan(hl) or hl <= 0:
                hl = 500
        except:
            hl = 500

        out["F"] = [pvalue, -volatility, hl]

In [5]:
def find_optimal_pairs(universe, formation_period=1000, trading_period=200, start_date=datetime(2015, 1, 1)):
    universe_period = get_universe_by_period(universe, formation_period, trading_period, start_date)
    optimal_pairs = []
    sectors = universe["GICS Sector"].unique()
    
    for period_df in tqdm(universe_period, desc="Finding optimal pairs"):
        print(f"Processing period: {period_df['period_start'].date()} to {period_df['trading_end'].date()}")

        period_dict = {
            "period_start": str(period_df["period_start"]),
            "formation_end": str(period_df["formation_end"]),
            "trading_start" : str(period_df["trading_start"]),
            "trading_end": str(period_df["trading_end"]),
            "optimal_pairs":[]
            }


        for sector in sectors:
            print(f"  Processing sector: {sector}")
            tickers = period_df[sector]
            if len(tickers) < 2:
                continue
            
            price_data = []
            valid_tickers = []
            
            for ticker in tickers:
                # Use only the formation window to select pairs (avoid look-ahead bias).
                data = get_data_from_cache(ticker, period_df["period_start"], period_df["formation_end"])
                if data is None:
                    continue

                price_col = "Adj Close" if "Adj Close" in data.columns else ("Close" if "Close" in data.columns else None)
                if price_col is None:
                    continue

                series = data[price_col]
                if series is None or series.empty:
                    continue

                price_data.append(series)
                valid_tickers.append(ticker)
            print(len(price_data), "valid price series found for sector", sector)
            if len(price_data) < 2:
                continue
            
            price_df = pd.concat(price_data, axis=1)
            price_df.columns = valid_tickers
            price_df = price_df.dropna(axis=1)
            valid_tickers = price_df.columns.tolist()
            print(f"    Found {len(valid_tickers)} valid tickers in sector {sector} after filtering")
            if len(valid_tickers) < 2:
                continue
            
            problem = PairsTradingProblem(price_df)
            algorithm = NSGA2(pop_size=50, # taille de la population
                            sampling=IntegerRandomSampling(), # méthode de génération initiale (échantillonnage aléatoire d'entiers)
                            crossover=SBX(prob=0.9, vtype=float, repair=RoundingRepair()), #croisement SBX adapté aux variables entières
                            mutation=PM(prob=0.1, vtype=float, repair=RoundingRepair()), # mutation PM adaptée aux variables entières
                            eliminate_duplicates=True)
            
            res = minimize(problem, algorithm, ('n_gen', 80), verbose=False)
            print(f"  Found {len(res.X) if res.X is not None else 0} candidate pairs in sector {sector}")
            if res.X is not None:
                X_array = np.atleast_2d(res.X)
                F_array = np.atleast_2d(res.F)
                for variables, scores in zip(X_array, F_array):
                    # ON FORCERA MAINTENANT UN HALF-LIFE < 45 JOURS DES LA CREATION DES PAIRES (3e objectif)
                    if scores[0] < 0.05 and scores[2] < 45:  
                        idx1, idx2 = int(variables[0]), int(variables[1])
                        pair = tuple(sorted((valid_tickers[idx1], valid_tickers[idx2])))
                        # Tuples convertis en listes pour la sérialisation JSON plus tard
                        if list(pair) not in period_dict["optimal_pairs"]:
                            period_dict["optimal_pairs"].append(list(pair))
        optimal_pairs.append(period_dict)
        print(optimal_pairs)
    return optimal_pairs


optimal_pairs = json.load(open("ml_pt_results/optimal_pairs_2.json", "r"))

# ON FORCE LE RE-CALCUL DES PAIRES DIRECTEMENT DANS CETTE CELLULE
# optimal_pairs = find_optimal_pairs(universe)

# # On sauvegarde la nouvelle distribution de paires ultra-rapides pour ne pas avoir à tout refaire
# with open("ml_pt_results/optimal_pairs_2.json", "w") as f:
#     json.dump(optimal_pairs, f, indent=4)

In [6]:
# --- FILTRE POST-SÉLECTION STRICT ---
def filter_strict_pairs(optimal_pairs, target_half_life=150, adf_threshold=0.10):
    """
    Filtre post-sélection génétique. Exige : 
    1) Un Hedge Ratio formation OLS strictement positif.
    2) Une ADF p-value < adf_threshold. 
    3) Un Half-Life < target_half_life jours.
    """
    filtered_periods = []
    
    stats = {"total": 0, "drop_data": 0, "drop_hr": 0, "drop_adf": 0, "drop_hl": 0, "kept": 0}
    hl_collection = []
    
    for period_dict in tqdm(optimal_pairs, desc="Strict Filtering"):
        period_start = pd.Timestamp(period_dict["period_start"]).tz_localize(None)
        formation_end = pd.Timestamp(period_dict["formation_end"]).tz_localize(None)
        
        valid_pairs_for_period = []
        
        for pair in period_dict["optimal_pairs"]:
            stats["total"] += 1
            data1 = get_data_from_cache(pair[0], period_start, formation_end)
            data2 = get_data_from_cache(pair[1], period_start, formation_end)
            
            if data1 is None or data2 is None:
                stats["drop_data"] += 1; continue
                
            cols1 = [c for c in ["Adj Close", "Close"] if c in data1.columns]
            cols2 = [c for c in ["Adj Close", "Close"] if c in data2.columns]
            if not cols1 or not cols2:
                stats["drop_data"] += 1; continue
                
            pair_df = pd.concat([data1[cols1[0]], data2[cols2[0]]], axis=1, join="inner").dropna()
            if len(pair_df) < 100:
                stats["drop_data"] += 1; continue
                
            pair_df.columns = ["s1", "s2"]
            pair_df.index = pd.to_datetime(pair_df.index, errors="coerce").normalize()
            if isinstance(pair_df.index, pd.DatetimeIndex) and pair_df.index.tz is not None:
                pair_df.index = pair_df.index.tz_localize(None)
            pair_df = pair_df[~pair_df.index.duplicated(keep="last")].sort_index()

            # Calcul du Hedge Ratio
            model_form = sm.OLS(pair_df["s1"], sm.add_constant(pair_df["s2"])).fit()
            beta_form = model_form.params.iloc[1]
            const_form = model_form.params.iloc[0]
            
            # Filtre : Hedge Ratio positif
            if beta_form <= 0:
                stats["drop_hr"] += 1; continue
                
            # Calcul du spread "Statique"
            spread_static = pair_df["s1"] - (beta_form * pair_df["s2"] + const_form)
            
            # Filtre : Cointégration sur le spread réel
            try:
                adf_result = sm.tsa.adfuller(spread_static)
                if adf_result[1] > adf_threshold:
                    stats["drop_adf"] += 1; continue
            except Exception:
                stats["drop_adf"] += 1; continue
                
            # Filtre : Half-life de mean reversion
            try:
                hl = compute_half_life(spread_static)
                # On collecte le HL valide pour analyse, peu importe sa durée
                if not np.isnan(hl) and hl != np.inf:
                    hl_collection.append(hl)
                    
                if hl >= target_half_life or np.isnan(hl):
                    stats["drop_hl"] += 1; continue
            except Exception:
                stats["drop_hl"] += 1; continue
                
            # C'est une super paire !
            valid_pairs_for_period.append(pair)
            stats["kept"] += 1
            
        # Reconstruction d'un dico propre
        filtered_dict = period_dict.copy()
        filtered_dict["optimal_pairs"] = valid_pairs_for_period
        filtered_periods.append(filtered_dict)
        
    print(f"\n--- RÉSULTATS DU FILTRAGE ---")
    print(f"Total évalué     : {stats['total']}")
    print(f"Rejetées (Data)  : {stats['drop_data']}")
    print(f"Rejetées (HR <0) : {stats['drop_hr']}")
    print(f"Rejetées (ADF)   : {stats['drop_adf']}")
    print(f"Rejetées (HL>150): {stats['drop_hl']}")
    print(f"Paires conservées: {stats['kept']} ({stats['kept']/max(1, stats['total']):.1%})")
    
    if hl_collection:
        hl_series = pd.Series(hl_collection)
        print(f"\n--- DISTRIBUTION DES HALF-LIFES (Valid ADF & HR) ---")
        print(hl_series.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    
    return filtered_periods

# On relâche le Half-life à 150 pour voir la vraie distribution des paires de l'algorithme NSGA-2
strict_optimal_pairs = filter_strict_pairs(optimal_pairs, target_half_life=30, adf_threshold=0.10)

Strict Filtering: 100%|██████████| 7/7 [02:36<00:00, 22.29s/it]


--- RÉSULTATS DU FILTRAGE ---
Total évalué     : 482
Rejetées (Data)  : 0
Rejetées (HR <0) : 0
Rejetées (ADF)   : 31
Rejetées (HL>150): 82
Paires conservées: 369 (76.6%)

--- DISTRIBUTION DES HALF-LIFES (Valid ADF & HR) ---
count    451.000000
mean      23.681448
std        8.726489
min        5.882263
10%       14.738181
25%       17.812974
50%       22.362895
75%       27.965816
90%       33.701957
max       88.033159
dtype: float64


In [ ]:
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).ewm(alpha=1/window, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/window, adjust=False).mean()
    rs = np.where(loss == 0, 0, gain / loss)
    rsi = 100 - (100 / (1 + pd.Series(rs, index=series.index)))
    return rsi

def compute_training_features(spread, z_score, s1_prices, s2_prices, s1_volume, s2_volume, spy_prices, tnx_prices=None, target_horizon=5):
    """
    Features basées strictement sur un spread statique. 
    Aucune fuite de données (look-ahead) via normalization glissante.
    """
    features_df = pd.DataFrame(index=spread.index)
    features_df["z_score"] = z_score
    features_df["spread_t"] = spread
    
    # Indicateurs Techniques
    features_df["rsi"] = compute_rsi(spread)
    features_df["macd"] = spread.ewm(span=12, adjust=False).mean() - spread.ewm(span=26, adjust=False).mean()
    for w in [5, 10, 20]:
        features_df[f"sma_{w}"] = spread.rolling(w).mean()

    # Corrélation brute 
    features_df["rolling_corr_30"] = s1_prices.pct_change().rolling(30).corr(s2_prices.pct_change())
    
    # --- NOUVELLES FEATURES ---
    # 1. Volatilité réalisée à 10j du spread
    features_df["spread_vol_10"] = spread.rolling(10).std()
    
    # 2. Ratio de volume s1 / s2 (lissé sur 5 jours pour éviter trop de bruit)
    try:
        features_df["volume_ratio_5d"] = (s1_volume.rolling(5).mean() / (s2_volume.rolling(5).mean() + 1e-9)).fillna(1.0)
    except Exception:
        features_df["volume_ratio_5d"] = 1.0

    # 3. Régime de marché (rolling beta 20j du spread par rapport au SPY)
    spread_diff = spread.diff()
    if spy_prices is not None and not spy_prices.empty:
        spy_ret = spy_prices.pct_change()
        spy_ret = spy_ret[~spy_ret.index.duplicated(keep="last")]
        spy_ret_aligned = spy_ret.reindex(spread.index).ffill().fillna(0.0)
        
        cov = spread_diff.rolling(20).cov(spy_ret_aligned)
        var = spy_ret_aligned.rolling(20).var()
        features_df["spread_spy_beta_20"] = (cov / (var + 1e-9)).fillna(0.0)
    else:
        features_df["spread_spy_beta_20"] = 0.0

    if tnx_prices is not None and not tnx_prices.empty:
        tnx_aligned = tnx_prices.reindex(spread.index).ffill().fillna(0.0)
        features_df["tnx_level"] = tnx_aligned
    else:
        features_df["tnx_level"] = 0.0
    
    # CIBLES : PnL du Spread pur
    features_df[f"target_spread_delta_t+{target_horizon}"] = spread.shift(-target_horizon) - spread
    features_df[f"target_zdelta_t+{target_horizon}"] = z_score.shift(-target_horizon) - z_score
    
    # CIBLE DE CLASSIFICATION BINAIRE : Y a-t-il eu reversion vers la moyenne ?
    # On regarde si la valeur absolue du z-score en t+H est plus petite que celle actuelle.
    # Cela donne 1 (True) si le spread a convergé vers la moyenne, 0 (False) sinon.
    z_next = z_score.shift(-target_horizon)
    features_df[f"target_is_revert_t+{target_horizon}"] = (np.abs(z_next) < np.abs(z_score)).astype(int)
    
    features_df = features_df.replace([np.inf, -np.inf], np.nan).dropna()
    return features_df

In [ ]:
def forecast_spread(
    optimal_pairs,
    train_every=20,
    target_horizon=5,
    z_entry=2.0,
    z_exit=0.5,
    min_prob_revert=0.60, 
    debug_summary=True,
    ):
    results = []
    stats = { k: 0 for k in [
        "periods", "pairs_total", "pairs_no_data", "pairs_empty_aligned", 
        "pairs_short_formation", "pairs_no_trading_window", "pairs_empty_features",
        "pairs_no_test_idx", "pairs_short_train", "signals_long", "signals_short", 
        "signals_flat", "blocked_exit_zone", "blocked_not_extreme", 
        "blocked_pred_strength", "blocked_corr_breakdown"
    ]}

    for period_dict in tqdm(optimal_pairs):
        stats["periods"] += 1
        period_start = pd.Timestamp(period_dict["period_start"]).tz_localize(None)
        formation_end = pd.Timestamp(period_dict["formation_end"]).tz_localize(None)
        trading_start = pd.Timestamp(period_dict["trading_start"]).tz_localize(None)
        trading_end = pd.Timestamp(period_dict["trading_end"]).tz_localize(None)

        # Charger l'indice SPY et le T-Bond 10Y pour la période
        spy_prices = None
        tnx_prices = None
        try:
            spy_data = yf.Ticker("SPY").history(start=period_start, end=trading_end)
            if "Close" in spy_data.columns:
                spy_prices = spy_data["Close"].copy()
                spy_prices.index = pd.to_datetime(spy_prices.index).tz_localize(None).normalize()
                spy_prices = spy_prices[~spy_prices.index.duplicated(keep="last")]
            tnx_data = yf.Ticker("^TNX").history(start=period_start, end=trading_end)
            if "Close" in tnx_data.columns:
                tnx_prices = tnx_data["Close"].copy()
                tnx_prices.index = pd.to_datetime(tnx_prices.index).tz_localize(None).normalize()
                tnx_prices = tnx_prices[~tnx_prices.index.duplicated(keep="last")]
        except Exception:
            pass

        for pair in period_dict["optimal_pairs"]:
            stats["pairs_total"] += 1
            data1 = get_data_from_cache(pair[0], period_start, trading_end)
            data2 = get_data_from_cache(pair[1], period_start, trading_end)
            
            if data1 is None or data2 is None: continue
                
            cols1 = [c for c in ["Adj Close", "Close"] if c in data1.columns]
            cols2 = [c for c in ["Adj Close", "Close"] if c in data2.columns]
            if not cols1 or not cols2: continue

            # Extract volumes (par défaut à 1 si non dispo)
            vol1 = data1["Volume"] if "Volume" in data1.columns else pd.Series(1, index=data1.index)
            vol2 = data2["Volume"] if "Volume" in data2.columns else pd.Series(1, index=data2.index)

            pair_df = pd.concat([data1[cols1[0]], data2[cols2[0]], vol1, vol2], axis=1, join="inner").dropna()
            pair_df.columns = ["s1", "s2", "v1", "v2"]
            pair_df.index = pd.to_datetime(pair_df.index, errors="coerce").normalize()
            if isinstance(pair_df.index, pd.DatetimeIndex) and pair_df.index.tz is not None:
                pair_df.index = pair_df.index.tz_localize(None)
            pair_df = pair_df[pair_df.index.notna()].sort_index()

            formation_df = pair_df.loc[pair_df.index <= formation_end].copy()
            if len(formation_df) < 100:
                stats["pairs_short_formation"] += 1; continue
                
            model_form = sm.OLS(formation_df["s1"], sm.add_constant(formation_df["s2"])).fit()
            beta_form = model_form.params.iloc[1]
            const_form = model_form.params.iloc[0]
            
            spread_static = pair_df["s1"] - (beta_form * pair_df["s2"] + const_form)
            
            mean_form = spread_static.loc[:formation_end].mean()
            std_form = spread_static.loc[:formation_end].std()
            if std_form == 0: continue
            
            z_score_static = (spread_static - mean_form) / std_form
            
            features_all = compute_training_features(
                spread_static, z_score_static, pair_df["s1"], pair_df["s2"],
                pair_df["v1"], pair_df["v2"], spy_prices, tnx_prices=tnx_prices, target_horizon=target_horizon
            )
            
            if features_all.empty:
                stats["pairs_empty_features"] += 1; continue

            features_all.index = features_all.index.normalize()
            test_start_idx = features_all.index.searchsorted(trading_start, side="left")
            if test_start_idx <= 0 or test_start_idx >= len(features_all):
                stats["pairs_no_test_idx"] += 1; continue

            target_col = f"target_is_revert_t+{target_horizon}"
            feature_cols = [c for c in features_all.columns if "target_" not in c]

            for start in range(test_start_idx, len(features_all), train_every):
                X_train = features_all[feature_cols].iloc[:start]
                y_train = features_all[target_col].iloc[:start]

                if len(X_train) < 50:
                    stats["pairs_short_train"] += 1; continue

                end = min(start + train_every, len(features_all))
                X_test = features_all[feature_cols].iloc[start:end]
                y_test_zdelta = features_all[f"target_zdelta_t+{target_horizon}"].iloc[start:end]
                y_test_delta = features_all[f"target_spread_delta_t+{target_horizon}"].iloc[start:end]

                # Standardisation et PCA
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)
                
                try:
                    pca = PCA(n_components=0.95, random_state=42)
                    X_train_pca = pca.fit_transform(X_train_scaled)
                    X_test_pca = pca.transform(X_test_scaled)
                except Exception:
                    X_train_pca = X_train_scaled
                    X_test_pca = X_test_scaled

                model = RandomForestClassifier(n_estimators=50, max_depth=4, min_samples_leaf=5, random_state=42, n_jobs=-1)
                model.fit(X_train_pca, y_train)
                
                from sklearn.ensemble import RandomForestRegressor
                model_reg = RandomForestRegressor(n_estimators=50, max_depth=4, min_samples_leaf=5, random_state=42, n_jobs=-1)
                y_train_zdelta = features_all[f"target_zdelta_t+{target_horizon}"].iloc[:start]
                model_reg.fit(X_train_pca, y_train_zdelta)
                pred_dz_list = model_reg.predict(X_test_pca)
                
                if len(model.classes_) == 2:
                    proba_revert = model.predict_proba(X_test_pca)[:, 1]
                else:
                    cls = model.classes_[0]
                    # Si toutes les données d'entraînement appartiennent à la classe '0', la probabilité de '1' (revert) est 0.
                    proba_revert = np.ones(len(X_test_pca)) if cls == 1 else np.zeros(len(X_test_pca))

                for current_date, z_now, corr_30, prob_rev, pred_dz, actual_dz, actual_delta in zip(
                    X_test.index, X_test["z_score"].values, X_test["rolling_corr_30"].values,
                    proba_revert, pred_dz_list, y_test_zdelta.values, y_test_delta.values
                ):
                    actual_z_next = z_now + actual_dz
                    
                    if np.isnan(corr_30) or corr_30 < 0.2:
                        position = "FLAT"; stats["blocked_corr_breakdown"] += 1
                    elif abs(z_now) <= z_exit:
                        position = "FLAT"; stats["blocked_exit_zone"] += 1
                    elif abs(z_now) < 1.0 and ((z_now < 0 and pred_dz < 0) or (z_now > 0 and pred_dz > 0)):
                        position = "FLAT"; stats["blocked_exit_zone"] += 1
                    elif z_now <= -z_entry:
                        if prob_rev < min_prob_revert:
                            position = "WAIT"; stats["blocked_pred_strength"] += 1
                        else:
                            position = "LONG"; stats["signals_long"] += 1
                    elif z_now >= z_entry:
                        if prob_rev < min_prob_revert:
                            position = "WAIT"; stats["blocked_pred_strength"] += 1
                        else:
                            position = "SHORT"; stats["signals_short"] += 1
                    else:
                        position = "WAIT"; stats["blocked_not_extreme"] += 1

                    if position == "FLAT":
                        stats["signals_flat"] += 1

                    if position != "WAIT":
                        results.append({
                            "date": current_date,
                            "pair": pair,
                            "current_zscore": z_now,
                            "hedge_ratio": float(beta_form), 
                            "rolling_corr_30": corr_30,
                            f"prob_revert_t+{target_horizon}": prob_rev,
                            f"actual_zdelta_t+{target_horizon}": actual_dz,
                            f"actual_zscore_t+{target_horizon}": actual_z_next,
                            "actual_delta": actual_delta,
                            "position": position,
                        })

    if debug_summary: print("=== forecast_spread debug summary ==="); print(stats)
    return pd.DataFrame(results)

results = pd.read_csv("ml_pt_results/trading_signals_2.csv")

# # PASSAGE SUR STRICT_OPTIMAL_PAIRS : SEULES LES PAIRES RIGOUREUSEMENT CO-INTÉGRÉES SUR L'HORIZON DE FORMATION PASSENT !
# results = forecast_spread(
#     strict_optimal_pairs, train_every=5, target_horizon=10, z_exit=0.5, z_entry=2.0,
#     min_prob_revert=0.60, debug_summary=True
# )

In [12]:
# results.to_csv("ml_pt_results/trading_signals_2.csv", index=False)

In [9]:
target_horizon = 10

trade_results = results[results["position"] != "FLAT"].copy()
if not trade_results.empty:
    trade_results["is_win"] = (
        ((trade_results["position"] == "LONG") & (trade_results[f"actual_zscore_t+{target_horizon}"] > trade_results["current_zscore"]))
        |
        ((trade_results["position"] == "SHORT") & (trade_results[f"actual_zscore_t+{target_horizon}"] < trade_results["current_zscore"]))
    )
    print(trade_results[["position", "current_zscore", f"actual_zscore_t+{target_horizon}", "is_win"]].head())
else:
    print("No signal generated.")

    position  current_zscore  actual_zscore_t+10  is_win
6      SHORT        2.123262            1.458958    True
105    SHORT        3.478223            2.776802    True
106    SHORT        3.360708            2.748529    True
107    SHORT        3.110315            2.862436    True
108    SHORT        2.233089            1.977566    True


In [10]:
trade_results = trade_results.copy()
if not trade_results.empty:
    trade_results["is_win"] = (
        ((trade_results["position"] == "LONG") & (trade_results[f"actual_zscore_t+{target_horizon}"] > trade_results["current_zscore"])) |
        ((trade_results["position"] == "SHORT") & (trade_results[f"actual_zscore_t+{target_horizon}"] < trade_results["current_zscore"]))
    )

    print("=== DEBUG SIGNAL QUALITY ===")
    print(f"Nombre de signaux (LONG/SHORT) : {len(trade_results)}")
    print(f"Win-rate sur z-score t+{target_horizon}       : {trade_results['is_win'].mean():.1%}")
    # print(f"Moyenne predicted_z - current_z (LONG only) : {(trade_results[trade_results['position']=='LONG']['predicted_zscore_t+{target_horizon}'] - trade_results[trade_results['position']=='LONG']['current_zscore']).mean():.3f}")
    # print(f"Moyenne predicted_z - current_z (SHORT only): {(trade_results[trade_results['position']=='SHORT']['predicted_zscore_t+{target_horizon}'] - trade_results[trade_results['position']=='SHORT']['current_zscore']).mean():.3f}")

    # trade_results["pred_improvement"] = np.where(
    #     trade_results["position"] == "LONG",
    #     trade_results["predicted_zscore_t+{target_horizon}"] - trade_results["current_zscore"],
    #     trade_results["current_zscore"] - trade_results["predicted_zscore_t+{target_horizon}]"]
    # )
    # print(f"Corrélation prédiction vs actual improvement : {trade_results['pred_improvement'].corr(trade_results['actual_zscore_t+5'] - trade_results['current_zscore']):.3f}")

=== DEBUG SIGNAL QUALITY ===
Nombre de signaux (LONG/SHORT) : 7056
Win-rate sur z-score t+10       : 62.1%


In [ ]:
def _normalize_pair(value):
    import ast

    if isinstance(value, (tuple, list)) and len(value) == 2:
        return (str(value[0]), str(value[1]))
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (tuple, list)) and len(parsed) == 2:
                return (str(parsed[0]), str(parsed[1]))
        except Exception:
            pass
        if "|" in value:
            a, b = value.split("|", 1)
            return (a.strip(), b.strip())
    return None


def _estimate_hedge_ratio(price_df, hedge_lookback=252, end_date=None):
    if price_df is None or price_df.empty:
        return None

    if end_date is None:
        window = price_df.tail(hedge_lookback)
    else:
        window = price_df.loc[:end_date].tail(hedge_lookback)

    if len(window) < 30:
        return None

    try:
        hr = sm.OLS(window["s1"], sm.add_constant(window["s2"])).fit().params.iloc[1]
    except Exception:
        return None

    if not np.isfinite(hr):
        return None

    return float(hr)


def prepare_pair_data_from_signals(
    signals_df,
    hedge_lookback=252,
    hold_until_signal_change=True,
    ):
    required_cols = {"date", "pair", "position"}
    missing = required_cols - set(signals_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = signals_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()].copy()
    if isinstance(df["date"].dtype, pd.DatetimeTZDtype):
        df["date"] = df["date"].dt.tz_localize(None)
    df["date"] = df["date"].dt.normalize()

    df["pair_norm"] = df["pair"].apply(_normalize_pair)
    df = df[df["pair_norm"].notna()].copy()
    df = df[df["position"].isin(["LONG", "SHORT", "FLAT"])].copy()

    if df.empty:
        return {
            "signals": df,
            "pair_data": {},
            "desired_side_by_pair": {},
            "all_dates": [],
            "rejected_negative_hr": [],
        }

    start_date = df["date"].min() - pd.Timedelta(days=7)
    end_date = df["date"].max() + pd.Timedelta(days=7)

    unique_pairs = sorted(df["pair_norm"].drop_duplicates().tolist())
    pair_data = {}
    rejected_negative_hr = []

    for ticker1, ticker2 in unique_pairs:
        data1 = get_data_from_cache(ticker1, start_date, end_date)
        data2 = get_data_from_cache(ticker2, start_date, end_date)
        if data1 is None or data2 is None:
            continue

        c1 = "Adj Close" if "Adj Close" in data1.columns else ("Close" if "Close" in data1.columns else None)
        c2 = "Adj Close" if "Adj Close" in data2.columns else ("Close" if "Close" in data2.columns else None)
        if c1 is None or c2 is None:
            continue

        px = pd.concat([data1[c1], data2[c2]], axis=1, join="inner").dropna()
        if len(px) < 30:
            continue

        px.columns = ["s1", "s2"]
        px.index = pd.to_datetime(px.index, errors="coerce")
        px = px[px.index.notna()]
        if isinstance(px.index, pd.DatetimeIndex) and px.index.tz is not None:
            px.index = px.index.tz_localize(None)
        px.index = px.index.normalize()
        px = px[~px.index.duplicated(keep="last")].sort_index()

        pair_sig = df[df["pair_norm"] == (ticker1, ticker2)]
        first_sig_date = pair_sig["date"].min()
        
        # Use exact hedge ratio generated by ML model on the signal date to avoid estimation mismatches
        init_hr = pair_sig[pair_sig["date"] == first_sig_date]["hedge_ratio"].iloc[0] if "hedge_ratio" in df.columns else _estimate_hedge_ratio(px.loc[px.index < first_sig_date], hedge_lookback)

        if init_hr is None:
            continue
            
        hl_val = 30
        try:
            from utils import compute_half_life
            spread_static = px.loc[px.index < first_sig_date, "s1"] - (init_hr * px.loc[px.index < first_sig_date, "s2"])
            estimated_hl = compute_half_life(spread_static)
            if pd.notna(estimated_hl) and estimated_hl > 0 and estimated_hl != float('inf'):
                hl_val = min(150, estimated_hl)
        except Exception:
            pass

        if init_hr <= 0:
            rejected_negative_hr.append((ticker1, ticker2, init_hr))
            continue

        r1_next = px["s1"].pct_change().shift(-1).dropna()
        r2_next = px["s2"].pct_change().shift(-1).dropna()
        aligned_idx = r1_next.index.intersection(r2_next.index)
        if len(aligned_idx) == 0:
            continue

        pair_data[(ticker1, ticker2)] = {
            "prices": px,
            "r1_next": r1_next.loc[aligned_idx],
            "r2_next": r2_next.loc[aligned_idx],
            "hedge_ratio_init": init_hr,
            "half_life": hl_val,
        }

    df = df[df["pair_norm"].isin(pair_data.keys())].copy()
    if df.empty:
        raise ValueError("No usable signals after pair data prep (check negative or missing hedge ratios).")

    side_map = {"LONG": 1.0, "SHORT": -1.0, "FLAT": 0.0}
    df["desired_side"] = df["position"].map(side_map).astype(float)
    if "prob_revert_t+10" in df.columns:
        df["confidence"] = df["prob_revert_t+10"]
    elif "prob_revert_t+5" in df.columns:
        df["confidence"] = df["prob_revert_t+5"]
    else:
        df["confidence"] = 0.70

    desired_side_by_pair = {}
    confidence_by_pair = {}
    for pair, pair_df in df.groupby("pair_norm"):
        pair_df = pair_df.sort_values("date")
        sig_series = pair_df.set_index("date")["desired_side"]
        conf_series = pair_df.set_index("date")["confidence"]
        ret_idx = pair_data[pair]["r1_next"].index
        if hold_until_signal_change:
            desired = sig_series.reindex(ret_idx).ffill().fillna(0.0)
            confs = conf_series.reindex(ret_idx).ffill().fillna(0.70)
        else:
            desired = sig_series.reindex(ret_idx).fillna(0.0)
            confs = conf_series.reindex(ret_idx).fillna(0.70)
        desired_side_by_pair[pair] = desired
        confidence_by_pair[pair] = confs

    all_dates = sorted(set().union(*[v["r1_next"].index for v in pair_data.values()]))

    return {
        "signals": df,
        "pair_data": pair_data,
        "desired_side_by_pair": desired_side_by_pair,
        "confidence_by_pair": confidence_by_pair,
        "all_dates": all_dates,
        "rejected_negative_hr": rejected_negative_hr,
    }


def _close_position_state(pair, state, exit_date, exit_reason, tc_rate):
    close_cost = state["notional"] * tc_rate
    state["cost_accum"] += close_cost
    net_trade_pnl = state["gross_pnl_accum"] - state["cost_accum"]

    row = {
        "pair": f"{pair[0]}|{pair[1]}",
        "entry_date": state["entry_date"],
        "exit_date": exit_date,
        "side": "LONG" if state["side"] > 0 else "SHORT",
        "entry_notional_gross": state["notional"],
        "hedge_ratio_exit": state["hedge_ratio"],
        "gross_pnl": state["gross_pnl_accum"],
        "cost": state["cost_accum"],
        "net_pnl": net_trade_pnl,
        "holding_days": (exit_date - state["entry_date"]).days if state["entry_date"] is not None else np.nan,
        "exit_reason": exit_reason,
    }

    state["side"] = 0.0
    state["notional"] = 0.0
    state["entry_date"] = None
    state["gross_pnl_accum"] = 0.0
    state["cost_accum"] = 0.0
    state["bars_since_hr_update"] = 0

    return row, close_cost


def simulate_portfolio(
    prepared_data,
    initial_capital=100_000,
    max_gross_exposure=1.0,
    risk_per_trade=0.10,
    fee_bps=5,
    slippage_bps=2,
    min_trades_per_day=1,
    max_holding_days=30,
    max_loss_pct=0.15,
    hedge_lookback=252,
    hedge_reestimate_every=20,
    rebalance_open_positions=True,
    rebalance_tolerance=0.20,
    rebalance_frequency_days=5,
    max_portfolio_drawdown_stop=0.50,
    ):
    pair_data = prepared_data["pair_data"]
    desired_side_by_pair = prepared_data["desired_side_by_pair"]
    confidence_by_pair = prepared_data.get("confidence_by_pair", {})
    all_dates = prepared_data["all_dates"]

    if len(all_dates) == 0 or len(pair_data) == 0:
        eq = pd.DataFrame([{"date": pd.Timestamp.today().normalize(), "equity": float(initial_capital), "daily_pnl": 0.0, "daily_cost": 0.0, "n_active_positions": 0, "circuit_breaker": False}])
        return pd.DataFrame(), eq, False

    tc_rate = (fee_bps + slippage_bps) / 10_000.0
    capital = float(initial_capital)
    circuit_breaker_triggered = False

    positions = {
        pair: {
            "side": 0.0,
            "notional": 0.0,
            "entry_date": None,
            "gross_pnl_accum": 0.0,
            "cost_accum": 0.0,
            "hedge_ratio": pair_data[pair]["hedge_ratio_init"],
            "bars_since_hr_update": 0,
        }
        for pair in pair_data.keys()
    }

    trade_rows = []
    daily_rows = []

    for day_idx, date in enumerate(all_dates):
        desired_today = {
            pair: float(desired_ser.loc[date]) if date in desired_ser.index else 0.0
            for pair, desired_ser in desired_side_by_pair.items()
        }

        n_targets = sum(1 for s in desired_today.values() if s != 0.0)
        if n_targets > 0 and capital > 0:
            gross_budget = capital * max_gross_exposure
            target_notional = min(gross_budget / n_targets, capital * risk_per_trade)
            target_notional = max(float(target_notional), 0.0)
        else:
            target_notional = 0.0

        entry_allowed = n_targets >= min_trades_per_day
        daily_cost = 0.0
        daily_gross_pnl = 0.0

        # 1) Signal transitions + risk exits
        for pair in pair_data.keys():
            state = positions[pair]
            cur_side = state["side"]
            des_side = desired_today[pair]
            forced_exit_reason = None

            if cur_side != 0.0 and state["entry_date"] is not None:
                holding_days = (date - state["entry_date"]).days
                unrealized_net = state["gross_pnl_accum"] - state["cost_accum"]
                loss_pct = unrealized_net / max(state["notional"], 1e-12)

                pair_max_hold = pair_data[pair].get("half_life", 30) * 2.5
                time_hit = holding_days >= pair_max_hold
                loss_hit = loss_pct <= -max_loss_pct
                if time_hit and loss_hit:
                    forced_exit_reason = "time_and_loss_stop"
                    des_side = 0.0
                elif time_hit:
                    forced_exit_reason = "time_stop"
                    des_side = 0.0
                elif loss_hit:
                    forced_exit_reason = "loss_stop"
                    des_side = 0.0

            if cur_side == des_side:
                pass
            elif cur_side == 0.0 and des_side != 0.0:
                conf = confidence_by_pair.get(pair, pd.Series()).get(date, 0.70) if isinstance(confidence_by_pair.get(pair), pd.Series) else 0.70
                if conf < 0.65:
                    pair_risk = 0.05
                elif conf > 0.75:
                    pair_risk = 0.15
                else:
                    pair_risk = risk_per_trade
                    
                target_notional_pair = min(gross_budget / n_targets, capital * pair_risk) if n_targets > 0 else 0.0
                target_notional_pair = max(float(target_notional_pair), 0.0)

                if target_notional_pair > 0.0 and entry_allowed:
                    open_cost = target_notional_pair * tc_rate
                    daily_cost += open_cost
                    state["side"] = des_side
                    state["notional"] = target_notional_pair
                    state["entry_date"] = date
                    state["gross_pnl_accum"] = 0.0
                    state["cost_accum"] = open_cost
                    state["bars_since_hr_update"] = 0
            elif cur_side != 0.0 and des_side == 0.0:
                exit_reason = forced_exit_reason if forced_exit_reason is not None else "signal_to_flat"
                row, close_cost = _close_position_state(pair, state, date, exit_reason, tc_rate)
                daily_cost += close_cost
                trade_rows.append(row)
            else:
                row, close_cost = _close_position_state(pair, state, date, "signal_reverse", tc_rate)
                daily_cost += close_cost
                trade_rows.append(row)

                conf = confidence_by_pair.get(pair, pd.Series()).get(date, 0.70) if isinstance(confidence_by_pair.get(pair), pd.Series) else 0.70
                if conf < 0.65:
                    pair_risk = 0.05
                elif conf > 0.75:
                    pair_risk = 0.15
                else:
                    pair_risk = risk_per_trade
                
                target_notional_pair = min(gross_budget / n_targets, capital * pair_risk) if n_targets > 0 else 0.0
                target_notional_pair = max(float(target_notional_pair), 0.0)

                if target_notional_pair > 0.0 and entry_allowed:
                    open_cost = target_notional_pair * tc_rate
                    daily_cost += open_cost
                    state["side"] = des_side
                    state["notional"] = target_notional_pair
                    state["entry_date"] = date
                    state["gross_pnl_accum"] = 0.0
                    state["cost_accum"] = open_cost
                    state["bars_since_hr_update"] = 0

        # 2) Optional scheduled rebalance
        rebalance_today = (
            rebalance_open_positions
            and target_notional > 0.0
            and rebalance_frequency_days > 0
            and (day_idx % rebalance_frequency_days == 0)
        )

        if rebalance_today:
            for pair in pair_data.keys():
                state = positions[pair]
                if state["side"] == 0.0 or state["notional"] <= 0.0:
                    continue
                rel_gap = abs(target_notional - state["notional"]) / max(state["notional"], 1e-12)
                if rel_gap >= rebalance_tolerance:
                    turnover = abs(target_notional - state["notional"])
                    rebalance_cost = turnover * tc_rate
                    daily_cost += rebalance_cost
                    state["cost_accum"] += rebalance_cost
                    state["notional"] = target_notional

        # 3) Mark-to-market + rolling hedge update
        for pair in pair_data.keys():
            state = positions[pair]
            if state["side"] == 0.0 or state["notional"] <= 0.0:
                continue

            if hedge_reestimate_every > 0 and state["bars_since_hr_update"] >= hedge_reestimate_every:
                updated_hr = _estimate_hedge_ratio(pair_data[pair]["prices"], hedge_lookback=hedge_lookback, end_date=date)
                if updated_hr is not None and updated_hr > 0:
                    state["hedge_ratio"] = updated_hr
                    state["bars_since_hr_update"] = 0

            r1_ser = pair_data[pair]["r1_next"]
            r2_ser = pair_data[pair]["r2_next"]
            if date not in r1_ser.index or date not in r2_ser.index:
                continue

            r1 = float(r1_ser.loc[date])
            r2 = float(r2_ser.loc[date])
            side = state["side"]
            h = state["hedge_ratio"]

            w1 = 1.0 / (1.0 + h)
            w2 = h / (1.0 + h)
            n1 = state["notional"] * w1
            n2 = state["notional"] * w2

            pnl_leg1 = n1 * side * r1
            pnl_leg2 = n2 * (-side) * r2
            pnl = pnl_leg1 + pnl_leg2

            daily_gross_pnl += pnl
            state["gross_pnl_accum"] += pnl
            state["bars_since_hr_update"] += 1

        daily_net_pnl = daily_gross_pnl - daily_cost
        capital = max(capital + daily_net_pnl, 0.0)

        n_active = sum(1 for st in positions.values() if st["side"] != 0.0)
        daily_rows.append({
            "date": date,
            "equity": capital,
            "daily_pnl": daily_net_pnl,
            "daily_cost": daily_cost,
            "n_active_positions": n_active,
            "circuit_breaker": False,
        })

        # 4) Portfolio-level circuit breaker
        breaker_level = initial_capital * (1.0 - max_portfolio_drawdown_stop)
        if capital <= breaker_level:
            breaker_close_cost = 0.0
            for pair, state in positions.items():
                if state["side"] == 0.0 or state["notional"] <= 0.0:
                    continue
                row, close_cost = _close_position_state(pair, state, date, "circuit_breaker", tc_rate)
                breaker_close_cost += close_cost
                trade_rows.append(row)

            if breaker_close_cost > 0.0 and len(daily_rows) > 0:
                capital = max(capital - breaker_close_cost, 0.0)
                daily_rows[-1]["daily_pnl"] -= breaker_close_cost
                daily_rows[-1]["daily_cost"] += breaker_close_cost
                daily_rows[-1]["equity"] = capital
                daily_rows[-1]["n_active_positions"] = 0

            daily_rows[-1]["circuit_breaker"] = True
            circuit_breaker_triggered = True
            break

    # 5) End-of-backtest liquidation
    if not circuit_breaker_triggered:
        last_date = all_dates[-1]
        final_close_cost = 0.0
        for pair, state in positions.items():
            if state["side"] == 0.0 or state["notional"] <= 0.0:
                continue
            row, close_cost = _close_position_state(pair, state, last_date, "end_of_backtest", tc_rate)
            final_close_cost += close_cost
            trade_rows.append(row)

        if final_close_cost > 0.0 and len(daily_rows) > 0:
            capital = max(capital - final_close_cost, 0.0)
            daily_rows[-1]["daily_pnl"] -= final_close_cost
            daily_rows[-1]["daily_cost"] += final_close_cost
            daily_rows[-1]["equity"] = capital
            daily_rows[-1]["n_active_positions"] = 0

    trades_df = pd.DataFrame(trade_rows).sort_values(["entry_date", "pair"]).reset_index(drop=True) if trade_rows else pd.DataFrame()
    equity_curve = pd.DataFrame(daily_rows).sort_values("date").reset_index(drop=True) if daily_rows else pd.DataFrame()

    if not equity_curve.empty:
        equity_curve["daily_return"] = equity_curve["equity"].pct_change().fillna(0.0)
        equity_curve["cum_return"] = equity_curve["equity"] / float(initial_capital) - 1.0
        equity_curve["rolling_max"] = equity_curve["equity"].cummax()
        equity_curve["drawdown"] = equity_curve["equity"] / equity_curve["rolling_max"] - 1.0

    return trades_df, equity_curve, circuit_breaker_triggered


def compute_backtest_metrics(trades_df, equity_curve, initial_capital=100_000):
    metrics = {
        "total_return": np.nan,
        "max_drawdown": np.nan,
        "sharpe": np.nan,
        "sortino": np.nan,
        "calmar": np.nan,
        "win_rate": np.nan,
        "avg_holding_days": np.nan,
        "closed_trades": 0,
    }

    if not equity_curve.empty:
        total_return = equity_curve["equity"].iloc[-1] / float(initial_capital) - 1.0
        max_dd = equity_curve["drawdown"].min()
        daily_ret = equity_curve["daily_return"]

        sharpe = np.nan
        if daily_ret.std() and daily_ret.std() > 0:
            sharpe = (daily_ret.mean() / daily_ret.std()) * np.sqrt(252)

        sortino = np.nan
        downside = daily_ret[daily_ret < 0]
        if downside.std() and downside.std() > 0:
            sortino = (daily_ret.mean() / downside.std()) * np.sqrt(252)

        calmar = np.nan
        if max_dd < 0:
            calmar = total_return / abs(max_dd)

        metrics.update({
            "total_return": total_return,
            "max_drawdown": max_dd,
            "sharpe": sharpe,
            "sortino": sortino,
            "calmar": calmar,
        })

    if isinstance(trades_df, pd.DataFrame) and not trades_df.empty:
        metrics["closed_trades"] = len(trades_df)
        metrics["win_rate"] = (trades_df["net_pnl"] > 0).mean()
        metrics["avg_holding_days"] = trades_df["holding_days"].mean()
        metrics["pnl_by_exit_reason"] = trades_df.groupby("exit_reason")["net_pnl"].agg(["count", "mean", "sum"])
    else:
        metrics["pnl_by_exit_reason"] = pd.DataFrame()

    return metrics


def print_backtest_summary(metrics, equity_curve, rejected_negative_hr=None, circuit_breaker_triggered=False):
    if equity_curve.empty:
        print("No equity curve data.")
        return

    print(f"Final equity: {equity_curve['equity'].iloc[-1]:,.2f}")
    print(f"Total return: {metrics['total_return']:.2%}" if np.isfinite(metrics["total_return"]) else "Total return: nan")
    print(f"Max drawdown: {metrics['max_drawdown']:.2%}" if np.isfinite(metrics["max_drawdown"]) else "Max drawdown: nan")
    print(f"Sharpe: {metrics['sharpe']:.3f}" if np.isfinite(metrics["sharpe"]) else "Sharpe: nan")
    print(f"Sortino: {metrics['sortino']:.3f}" if np.isfinite(metrics["sortino"]) else "Sortino: nan")
    print(f"Calmar: {metrics['calmar']:.3f}" if np.isfinite(metrics["calmar"]) else "Calmar: nan")
    print(f"Closed trades: {metrics['closed_trades']}")
    print(f"Trade win rate: {metrics['win_rate']:.2%}" if np.isfinite(metrics["win_rate"]) else "Trade win rate: nan")
    print(f"Average holding days: {metrics['avg_holding_days']:.2f}" if np.isfinite(metrics["avg_holding_days"]) else "Average holding days: nan")

    if rejected_negative_hr:
        print(f"Rejected pairs with non-positive hedge ratio: {len(rejected_negative_hr)}")

    if circuit_breaker_triggered:
        print("Circuit breaker triggered: trading stopped early.")

    pnl_by_exit = metrics.get("pnl_by_exit_reason", pd.DataFrame())
    if isinstance(pnl_by_exit, pd.DataFrame) and not pnl_by_exit.empty:
        print("Net PnL by exit reason:")
        print(pnl_by_exit)


def build_equity_curve_from_signals(
    signals_df,
    initial_capital=100_000,
    max_gross_exposure=1.0,
    risk_per_trade=0.10,
    fee_bps=5,
    slippage_bps=2,
    min_trades_per_day=1,
    hold_until_signal_change=True,
    max_holding_days=30,
    max_loss_pct=0.08,
    hedge_lookback=252,
    hedge_reestimate_every=20,
    rebalance_open_positions=True,
    rebalance_tolerance=0.20,
    rebalance_frequency_days=5,
    max_portfolio_drawdown_stop=0.50,
    verbose=True,
    return_metrics=False,
    ):
    prepared = prepare_pair_data_from_signals(
        signals_df,
        hedge_lookback=hedge_lookback,
        hold_until_signal_change=hold_until_signal_change,
    )

    trades_df, equity_curve, circuit_breaker_triggered = simulate_portfolio(
        prepared_data=prepared,
        initial_capital=initial_capital,
        max_gross_exposure=max_gross_exposure,
        risk_per_trade=risk_per_trade,
        fee_bps=fee_bps,
        slippage_bps=slippage_bps,
        min_trades_per_day=min_trades_per_day,
        max_holding_days=max_holding_days,
        max_loss_pct=max_loss_pct,
        hedge_lookback=hedge_lookback,
        hedge_reestimate_every=hedge_reestimate_every,
        rebalance_open_positions=rebalance_open_positions,
        rebalance_tolerance=rebalance_tolerance,
        rebalance_frequency_days=rebalance_frequency_days,
        max_portfolio_drawdown_stop=max_portfolio_drawdown_stop,
    )

    metrics = compute_backtest_metrics(trades_df, equity_curve, initial_capital=initial_capital)
    if verbose:
        print_backtest_summary(
            metrics,
            equity_curve,
            rejected_negative_hr=prepared["rejected_negative_hr"],
            circuit_breaker_triggered=circuit_breaker_triggered,
        )

    if return_metrics:
        return trades_df, equity_curve, metrics
    return trades_df, equity_curve


trades_df, equity_curve = build_equity_curve_from_signals(
    results,
    initial_capital=100_000,
    max_gross_exposure=1,
    risk_per_trade=0.1,
    fee_bps=2, 
    slippage_bps=2,
    min_trades_per_day=1,
    hold_until_signal_change=True, # CRITIQUE : Forward-fill activé
    max_holding_days=30, # STOP TIME repoussé pour laisser converger
    max_loss_pct=0.2,
    hedge_lookback=252,
    hedge_reestimate_every=0,
    rebalance_open_positions=False,
    rebalance_tolerance=0.20,
    rebalance_frequency_days=10,
    max_portfolio_drawdown_stop=0.50,
    verbose=True,
    )

fig = px.line(equity_curve, x="date", y="equity", title="Equity Curve (Modular Backtest)")
fig.show()

equity_curve.tail()

Final equity: 109,148.40
Total return: 9.15%
Max drawdown: -3.92%
Sharpe: 0.506
Sortino: 0.823
Calmar: 2.335
Closed trades: 6157
Trade win rate: 50.74%
Average holding days: 28.81
Net PnL by exit reason:
                 count       mean           sum
exit_reason                                    
end_of_backtest    163  -2.341538   -381.670671
signal_reverse      10  88.194068    881.940679
signal_to_flat     482  30.772610  14832.397995
time_stop         5502  -1.124003  -6184.264847


,date,equity,daily_pnl,daily_cost,n_active_positions,circuit_breaker,daily_return,cum_return,rolling_max,drawdown
1391,2024-06-27,109404.564620,-10.579856,0.276647,162,False,-0.000097,0.094046,113471.598549,-0.035842
1392,2024-06-28,109262.607939,-141.956681,11.050750,126,False,-0.001298,0.092626,113471.598549,-0.037093
1393,2024-07-01,109283.155962,20.548023,11.138767,164,False,0.000188,0.092832,113471.598549,-0.036912
1394,2024-07-02,109229.207381,-53.948581,0.264929,165,False,-0.000494,0.092292,113471.598549,-0.037387
1395,2024-07-03,109148.403157,-80.804224,44.296847,0,False,-0.000740,0.091484,113471.598549,-0.038099


In [34]:
trades_df

,pair,entry_date,exit_date,side,entry_notional_gross,hedge_ratio_exit,gross_pnl,cost,net_pnl,holding_days,exit_reason
0,DUK|SRE,2018-12-27,2019-01-28,SHORT,20000.000000,1.384540,705.169777,16.000000,689.169777,32,time_stop
1,BXP|IRM,2019-01-02,2019-01-25,LONG,20024.172071,1.059706,7.567776,16.019338,-8.451562,23,signal_to_flat
2,AEP|XEL,2019-01-29,2019-02-28,SHORT,20136.143643,1.229034,336.703204,16.108915,320.594289,30,time_stop
3,DUK|SRE,2019-01-29,2019-02-28,SHORT,20136.143643,1.384540,187.188284,16.108915,171.079369,30,time_stop
4,ED|NI,2019-01-30,2019-02-20,LONG,20154.021259,2.356321,406.131751,16.123217,390.008534,21,signal_to_flat
...,...,...,...,...,...,...,...,...,...,...,...
3832,MCO|V,2024-07-12,2024-07-12,LONG,1114.590579,2.264584,-7.333716,0.891672,-8.225389,0,end_of_backtest
3833,MET|PNC,2024-07-12,2024-07-12,SHORT,1114.590579,0.308835,-8.822777,0.891672,-9.714449,0,end_of_backtest
3834,MLM|VMC,2024-07-12,2024-07-12,LONG,1114.590579,2.460569,-3.171733,0.891672,-4.063406,0,end_of_backtest
3835,NWS|NWSA,2024-07-12,2024-07-12,LONG,1114.590579,0.980122,0.200173,0.891672,-0.691500,0,end_of_backtest


In [52]:
if not trade_results.empty:
    trade_results["actual_move"] = trade_results["actual_zscore_t+5"] - trade_results["current_zscore"]
    trade_results["signed_move"] = np.where(
        trade_results["position"] == "LONG",
        trade_results["actual_move"],
        -trade_results["actual_move"]
    )

    wins = trade_results[trade_results["is_win"] == True]["signed_move"]
    losses = trade_results[trade_results["is_win"] == False]["signed_move"]

    if len(wins) > 0 and len(losses) > 0:
        print(f"Gain moyen quand correct  : +{wins.mean():.4f}")
        print(f"Perte moyenne quand faux  : {losses.mean():.4f}")
        print(f"Ratio perte/gain          : {abs(losses.mean()/wins.mean()):.2f}x")
    else:
        print("Not enough wins or losses to compute ratio.")

Gain moyen quand correct  : +0.7697
Perte moyenne quand faux  : -0.5759
Ratio perte/gain          : 0.75x
